# Lightweight Fine-Tuning Project

In this project we use

PEFT technique: LoRA, as it enables efficient fine-tuning for LLMs.
Model: DistilBERT, which allows for rapid fine-tuning with minimal hardware resources.
Evaluation approach: precision and loss metrics.
Fine-tuning dataset: SMS Spam, as it aligns with the project’s requirements of using small, easy-to-handle datasets.

## Loading and Evaluating a Foundation Model

TODO: In the cells below, load your chosen pre-trained Hugging Face model and evaluate its performance prior to fine-tuning. This step includes loading an appropriate tokenizer and dataset.

In [48]:
from transformers import AutoModelForSequenceClassification, AutoTokenizer, Trainer, TrainingArguments, DataCollatorWithPadding
from datasets import load_dataset
import numpy as np

In [49]:
dataset = load_dataset("sms_spam")
dataset = dataset["train"].train_test_split(test_size=0.2, seed=42)

In [50]:
id2label = {0: "NOT SPAM", 1: "SPAM"}
label2id = {"NOT SPAM": 0, "SPAM": 1}

print("Ejemplos del conjunto de entrenamiento:")
for entry in dataset["train"].select(range(3)):
    print(f"label={id2label[entry['label']]}, sms={entry['sms']}")

print("\nEjemplos del conjunto de prueba:")
for entry in dataset["test"].select(range(3)):
    print(f"label={id2label[entry['label']]}, sms={entry['sms']}")

Ejemplos del conjunto de entrenamiento:
label=NOT SPAM, sms=K..then come wenever u lik to come and also tel vikky to come by getting free time..:-)

label=NOT SPAM, sms=:-) :-)

label=NOT SPAM, sms=I need you to be in my strong arms...


Ejemplos del conjunto de prueba:
label=SPAM, sms=sports fans - get the latest sports news str* 2 ur mobile 1 wk FREE PLUS a FREE TONE Txt SPORT ON to 8007 www.getzed.co.uk 0870141701216+ norm 4txt/120p 

label=NOT SPAM, sms=It's justbeen overa week since we broke up and already our brains are going to mush!

label=NOT SPAM, sms=Not directly behind... Abt 4 rows behind ü...



In [51]:
def contar_etiquetas(ds, nombre):
    labels = ds["label"]
    total = len(labels)
    spam = sum(1 for l in labels if l == 1)
    not_spam = total - spam
    print(f"{nombre}: total={total}, SPAM={spam}, NOT SPAM={not_spam}")

contar_etiquetas(dataset["train"], "Train")
contar_etiquetas(dataset["test"], "Test")

Train: total=4459, SPAM=598, NOT SPAM=3861
Test: total=1115, SPAM=149, NOT SPAM=966


In [52]:
model = AutoModelForSequenceClassification.from_pretrained(
    "distilbert-base-uncased",
    num_labels=2,
    id2label=id2label,
    label2id=label2id,
)

print(model)

# Contar parámetros totales y entrenables
total_params = sum(p.numel() for p in model.parameters())
trainable_params = sum(p.numel() for p in model.parameters() if p.requires_grad)

print(f"Total parámetros: {total_params}")
print(f"Parámetros entrenables: {trainable_params}")


Some weights of DistilBertForSequenceClassification were not initialized from the model checkpoint at distilbert-base-uncased and are newly initialized: ['classifier.bias', 'classifier.weight', 'pre_classifier.bias', 'pre_classifier.weight']
You should probably TRAIN this model on a down-stream task to be able to use it for predictions and inference.


DistilBertForSequenceClassification(
  (distilbert): DistilBertModel(
    (embeddings): Embeddings(
      (word_embeddings): Embedding(30522, 768, padding_idx=0)
      (position_embeddings): Embedding(512, 768)
      (LayerNorm): LayerNorm((768,), eps=1e-12, elementwise_affine=True, bias=True)
      (dropout): Dropout(p=0.1, inplace=False)
    )
    (transformer): Transformer(
      (layer): ModuleList(
        (0-5): 6 x TransformerBlock(
          (attention): DistilBertSdpaAttention(
            (dropout): Dropout(p=0.1, inplace=False)
            (q_lin): Linear(in_features=768, out_features=768, bias=True)
            (k_lin): Linear(in_features=768, out_features=768, bias=True)
            (v_lin): Linear(in_features=768, out_features=768, bias=True)
            (out_lin): Linear(in_features=768, out_features=768, bias=True)
          )
          (sa_layer_norm): LayerNorm((768,), eps=1e-12, elementwise_affine=True, bias=True)
          (ffn): FFN(
            (dropout): Dropout(

In [53]:
tokenizer = AutoTokenizer.from_pretrained("distilbert-base-uncased")

def tokenize(batch):
    return tokenizer(batch["sms"], padding="max_length", truncation=True)

tokenized_ds = dataset.map(tokenize, batched=True)

In [54]:
pip install scikit-learn

Note: you may need to restart the kernel to use updated packages.


In [55]:
from sklearn.metrics import accuracy_score, precision_recall_fscore_support

In [56]:
def compute_metrics(eval_pred):
    logits, labels = eval_pred
    predictions = logits.argmax(axis=-1)
    
    acc = accuracy_score(labels, predictions)
    precision, recall, f1, _ = precision_recall_fscore_support(
        labels, predictions, average="binary"
    )
    
    return {
        "accuracy": acc,
        "precision": precision,
        "recall": recall,
        "f1": f1,
    }

training_args = TrainingArguments(
    output_dir="./results",
    per_device_eval_batch_size=16,
)

trainer = Trainer(
    model=model,
    args=training_args,
    eval_dataset=tokenized_ds["test"],
    tokenizer=tokenizer,
    compute_metrics=compute_metrics,
)

baseline_results = trainer.evaluate()
print("\nBaseline evaluation:", baseline_results)

C:\Users\amspo\AppData\Local\Temp\ipykernel_5860\951788056.py:22: FutureWarning: `tokenizer` is deprecated and will be removed in version 5.0.0 for `Trainer.__init__`. Use `processing_class` instead.
  trainer = Trainer(



Baseline evaluation: {'eval_loss': 0.6846591234207153, 'eval_model_preparation_time': 0.001, 'eval_accuracy': 0.7183856502242153, 'eval_precision': 0.2727272727272727, 'eval_recall': 0.6644295302013423, 'eval_f1': 0.38671875, 'eval_runtime': 10.4437, 'eval_samples_per_second': 106.763, 'eval_steps_per_second': 6.703}


## Performing Parameter-Efficient Fine-Tuning

TODO: In the cells below, create a PEFT model from your loaded model, run a training loop, and save the PEFT model weights.

In [57]:
from peft import LoraConfig, get_peft_model, TaskType, PeftModel, AutoPeftModelForSequenceClassification

In [58]:
lora_config = LoraConfig(
    task_type=TaskType.SEQ_CLS,
    r=8,
    lora_alpha=16,
    lora_dropout=0.1,
    bias="none",
    target_modules=["q_lin","v_lin"]
)

peft_model = get_peft_model(model, lora_config)

In [59]:
total_params = sum(p.numel() for p in peft_model.parameters())
trainable_params = sum(p.numel() for p in peft_model.parameters() if p.requires_grad)
print(f"Total parámetros: {total_params:,}")
print(f"Parámetros entrenables: {trainable_params:,}")
print(f"Porcentaje entrenable: {100 * trainable_params / total_params:.2f}%")

Total parámetros: 67,694,596
Parámetros entrenables: 739,586
Porcentaje entrenable: 1.09%


In [60]:
trainer = Trainer(
    model=peft_model,
    args=TrainingArguments(
        output_dir="./data/sms_spam_lora",
        learning_rate=2e-3,
        per_device_train_batch_size=4,
        per_device_eval_batch_size=4,
        num_train_epochs=2,
        weight_decay=0.01,
        eval_strategy="epoch",
        save_strategy="epoch",
        load_best_model_at_end=True,
    ),
    train_dataset=tokenized_ds["train"],
    eval_dataset=tokenized_ds["test"],
    tokenizer=tokenizer,
    data_collator=DataCollatorWithPadding(tokenizer=tokenizer),
    compute_metrics=compute_metrics,
)

trainer.train()

C:\Users\amspo\AppData\Local\Temp\ipykernel_5860\1181708186.py:1: FutureWarning: `tokenizer` is deprecated and will be removed in version 5.0.0 for `Trainer.__init__`. Use `processing_class` instead.
  trainer = Trainer(


Epoch,Training Loss,Validation Loss,Accuracy,Precision,Recall,F1
1,0.109100,0.056368,0.986547,0.952703,0.946309,0.949495
2,0.039000,0.047980,0.991928,0.986111,0.953020,0.969283


TrainOutput(global_step=2230, training_loss=0.09294643124122791, metrics={'train_runtime': 237.2857, 'train_samples_per_second': 37.583, 'train_steps_per_second': 9.398, 'total_flos': 1201606030270464.0, 'train_loss': 0.09294643124122791, 'epoch': 2.0})

###  ⚠️ IMPORTANT ⚠️

Due to workspace storage constraints, you should not store the model weights in the same directory but rather use `/tmp` to avoid workspace crashes which are irrecoverable.
Ensure you save it in /tmp always.

In [61]:
# Saving the model
peft_model.save_pretrained("/tmp/sms_spam_lora")

## Performing Inference with a PEFT Model

TODO: In the cells below, load the saved PEFT model weights and evaluate the performance of the trained PEFT model. Be sure to compare the results to the results from prior to fine-tuning.

In [62]:
loaded_peft_model = AutoPeftModelForSequenceClassification.from_pretrained(
    "/tmp/sms_spam_lora",  # ajusta al checkpoint que tengas
    num_labels=2,
    id2label=id2label,
    label2id=label2id,
)

Some weights of DistilBertForSequenceClassification were not initialized from the model checkpoint at distilbert-base-uncased and are newly initialized: ['classifier.bias', 'classifier.weight', 'pre_classifier.bias', 'pre_classifier.weight']
You should probably TRAIN this model on a down-stream task to be able to use it for predictions and inference.


In [63]:
trainer = Trainer(
    model=loaded_peft_model,
    args=TrainingArguments(
        output_dir="./data/sms_spam_lora",
        learning_rate=2e-3,
        per_device_train_batch_size=4,
        per_device_eval_batch_size=4,
        num_train_epochs=2,
        weight_decay=0.01,
        eval_strategy="epoch",
        save_strategy="epoch",
        load_best_model_at_end=True,
    ),
    train_dataset=tokenized_ds["train"],
    eval_dataset=tokenized_ds["test"],
    tokenizer=tokenizer,
    data_collator=DataCollatorWithPadding(tokenizer=tokenizer),
    compute_metrics=compute_metrics,
)

C:\Users\amspo\AppData\Local\Temp\ipykernel_5860\1094218539.py:1: FutureWarning: `tokenizer` is deprecated and will be removed in version 5.0.0 for `Trainer.__init__`. Use `processing_class` instead.
  trainer = Trainer(


In [64]:
results_lora = trainer.evaluate()
print("Resultados del modelo LoRA:", results_lora)

Resultados del modelo LoRA: {'eval_loss': 0.04798038303852081, 'eval_model_preparation_time': 0.0102, 'eval_accuracy': 0.9919282511210762, 'eval_precision': 0.9861111111111112, 'eval_recall': 0.9530201342281879, 'eval_f1': 0.9692832764505119, 'eval_runtime': 11.4572, 'eval_samples_per_second': 97.319, 'eval_steps_per_second': 24.352}
